# YoutubeAgents Realistic Audio Generation Pipeline
Chạy mô hình **AudioLDM 2 (Large)** trên GPU Tesla T4 để tạo âm thanh Foley/SFX thực tế cho video.

In [ ]:
# Kaggle AudioLDM 2 Realistic Foley Generator
# Auto-generated by YoutubeAgents KaggleAudioProvider
import os
import sys
import json
import torch
import scipy.io.wavfile
import numpy as np

# 1. Install & import diffusers
os.system("pip install -q diffusers transformers accelerate torchaudio scipy")
from diffusers import AudioLDM2Pipeline

print("=== Starting Kaggle AudioLDM 2 Realistic Audio Engine ===")
print("Torch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "cvssp/audioldm2-large"
print(f"Loading model: {model_id}...")
repo_id = model_id
pipe = AudioLDM2Pipeline.from_pretrained(repo_id, torch_dtype=dtype)
pipe = pipe.to(device)
if torch.cuda.is_available():
    pipe.enable_attention_slicing()

out_dir = "/kaggle/working/output_audio"
os.makedirs(out_dir, exist_ok=True)

specs = [
  {
    "scene_index": 0,
    "prompt": "cinematic deep impact boom with low-end sub bass and subtle room decay",
    "negative_prompt": "distorted, low quality, noise, muffled, robotic, speech",
    "duration_seconds": 4.0,
    "guidance_scale": 4.0,
    "num_inference_steps": 40
  }
]

results = []
for idx, item in enumerate(specs):
    scene_idx = item["scene_index"]
    prompt = item["prompt"]
    neg_prompt = item["negative_prompt"]
    duration = item["duration_seconds"]
    guidance = item["guidance_scale"]
    steps = item["num_inference_steps"]
    
    print(f"[{idx+1}/{len(specs)}] Generating realistic Foley for Scene {scene_idx}: '{prompt}'...")
    
    audio = pipe(
        prompt=prompt,
        negative_prompt=neg_prompt,
        num_inference_steps=steps,
        audio_length_in_s=duration,
        guidance_scale=guidance,
    ).audios[0]

    out_file = os.path.join(out_dir, f"foley_scene_{scene_idx:02d}.wav")
    # Convert to 16-bit PCM WAV
    scipy.io.wavfile.write(out_file, rate=16000, data=audio)
    
    print(f"Saved: {out_file} (Size: {os.path.getsize(out_file)} bytes)")
    results.append({"scene_index": scene_idx, "file": out_file, "prompt": prompt})

with open(os.path.join(out_dir, "manifest.json"), "w") as f:
    json.dump(results, f, indent=2)

print("=== Kaggle Audio Generation Completed Successfully! ===")
